In [7]:
import csv
import os

# Caminhos corretos dos vídeos
video_paths = {
    "LEARN_COM": "C:/DataSet_SoccerKeyFrame/LEARN/LE_C_FOG/",
    "LEARN_Sem": "C:/DataSet_SoccerKeyFrame/LEARN/LE_S_FOG/",
    "CHECK_COM": "C:/DataSet_SoccerKeyFrame/CHECK/CH_C_FOG/",
    "CHECK_Sem": "C:/DataSet_SoccerKeyFrame/CHECK/CH_S_FOG/"
}

# Dados de exemplo
data = []
data.append(["video_path", "start_time", "end_time", "description"])

# Duração da propaganda
duration_seconds = 7
duration_frames = 22

# Função para calcular start_time e end_time a partir do time code
def calculate_times(time_part):
    minutes, seconds, frames = map(int, time_part.split('-'))

    start_frame = (minutes * 60 + seconds) * 30 + frames  # Supondo que 30 fps

    end_seconds = seconds + duration_seconds
    end_frames = frames + duration_frames

    if end_frames >= 30:
        end_seconds += end_frames // 30
        end_frames = end_frames % 30

    end_minutes = minutes + end_seconds // 60
    end_seconds = end_seconds % 60

    start_time = f"{minutes:02}:{seconds:02}:{frames:02}"
    end_time = f"{end_minutes:02}:{end_seconds:02}:{end_frames:02}"

    return start_frame, start_time, end_time

# Função para processar vídeos
def process_videos(video_path, with_ads):
    for video_name in os.listdir(video_path):
        if video_name.endswith(".mp4"):  # Processa apenas arquivos MP4
            if with_ads:  # Vídeos com propaganda
                if len(video_name) == 20:  # 16 caracteres + 4 para extensão
                    time_part = video_name[8:16]  # Pega os caracteres do time code
                    start_frame, start_time, end_time = calculate_times(time_part)
                    data.append([os.path.join(video_path, video_name), start_time, end_time, "Informar Patrocinador"])
                    print(f"[COM] Vídeo: {video_name} | Início: {start_time} | Fim: {end_time}")
                else:
                    print(f"[ERRO] Vídeo com propaganda mal formatado: {video_name}")
            else:  # Vídeos sem propaganda
                if len(video_name) == 14:  # 10 caracteres + 4 para extensão
                    data.append([os.path.join(video_path, video_name), None, None, "Sem propaganda"])
                    print(f"[SEM] Vídeo: {video_name} | Sem propaganda")
                else:
                    print(f"[ERRO] Vídeo sem propaganda mal formatado: {video_name}")

# Processando vídeos de todas as pastas
process_videos(video_paths["LEARN_COM"], True)
process_videos(video_paths["LEARN_Sem"], False)
process_videos(video_paths["CHECK_COM"], True)
process_videos(video_paths["CHECK_Sem"], False)

# Salvando arquivo CSV com o novo nome
with open('DataSet_SoccerKeyFrame.csv', 'w', newline='') as f:
    writer = csv.writer(f)
    writer.writerows(data)

print("Arquivo DataSet_SoccerKeyFrame.csv gerado com sucesso!")

[COM] Vídeo: CFOG01T_00-28-18.mp4 | Início: 00:28:18 | Fim: 00:36:10
[COM] Vídeo: CFOG03T_01-26-01.mp4 | Início: 01:26:01 | Fim: 01:33:23
[COM] Vídeo: CFOG05T_00-18-04.mp4 | Início: 00:18:04 | Fim: 00:25:26
[COM] Vídeo: CFOG06T_01-49-28.mp4 | Início: 01:49:28 | Fim: 01:57:20
[COM] Vídeo: CFOG08T_00-40-00.mp4 | Início: 00:40:00 | Fim: 00:47:22
[COM] Vídeo: CFOG10T_00-53-13.mp4 | Início: 00:53:13 | Fim: 01:01:05
[COM] Vídeo: CFOG11T_01-12-11.mp4 | Início: 01:12:11 | Fim: 01:20:03
[COM] Vídeo: CFOG13T_01-00-16.mp4 | Início: 01:00:16 | Fim: 01:08:08
[COM] Vídeo: CFOG15T_01-35-00.mp4 | Início: 01:35:00 | Fim: 01:42:22
[COM] Vídeo: CFOG16T_01-39-15.mp4 | Início: 01:39:15 | Fim: 01:47:07
[COM] Vídeo: CFOG18T_01-18-07.mp4 | Início: 01:18:07 | Fim: 01:25:29
[COM] Vídeo: CFOG20T_00-15-00.mp4 | Início: 00:15:00 | Fim: 00:22:22
[COM] Vídeo: CFOG21T_00-20-13.mp4 | Início: 00:20:13 | Fim: 00:28:05
[COM] Vídeo: CFOG23T_00-45-16.mp4 | Início: 00:45:16 | Fim: 00:53:08
[COM] Vídeo: CFOG25T_01-05-14.mp4 

In [8]:
pip install pandas openpyxl

Note: you may need to restart the kernel to use updated packages.


In [1]:
!pip install openpyxl
import pandas as pd
import os
import re

# Caminhos corretos dos vídeos
video_paths = {
    "TRAIN_COM": "C:/Vi_DataSet_S_K_Frame/TRAIN/LE_C_FOG/",
    "TRAIN_Sem": "C:/Vi_DataSet_S_K_Frame/TRAIN/LE_S_FOG/",
    "VALID": "C:/Vi_DataSet_S_K_Frame/VALID/",
    "TEST": "C:/Vi_DataSet_S_K_Frame/TEST/"
}
# ─── Defina o diretório de saída ─────────────────────────────────────
OUT_DIR = r"C:\DataSet_SoccerKeyFrame - Org"
os.makedirs(OUT_DIR, exist_ok=True)

# Lista para armazenar metadados
data = [[
    "video_path", "start_time", "end_time", "description", 
    "Duration", "Format", "Codec", "Resolution", "size_MB"
]]

# Parâmetros de duração de propaganda
DURATION_SECONDS = 7
DURATION_FRAMES = 22

# Metadados fixos do vídeo
DURATION = "2 min"
FORMAT = ".MP4"
CODEC = "H.264"
FRAME_SIZE = "1920 x 1080"

# Regex para identificar timecode no nome (mm-ss-ff)
timecode_pattern = re.compile(
    r"^(?P<prefix>.{8})(?P<time>\d{2}-\d{2}-\d{2})\.mp4$",
    re.IGNORECASE
)

# Função para converter timecode em tempos legíveis
def calculate_times(time_part: str):
    minutes, seconds, frames = map(int, time_part.split('-'))
    start_frame = (minutes * 60 + seconds) * 30 + frames

    end_seconds = seconds + DURATION_SECONDS
    end_frames = frames + DURATION_FRAMES
    if end_frames >= 30:
        end_seconds += end_frames // 30
        end_frames %= 30

    end_minutes = minutes + end_seconds // 60
    end_seconds %= 60

    start_time = f"{minutes:02}:{seconds:02}:{frames:02}"
    end_time = f"{end_minutes:02}:{end_seconds:02}:{end_frames:02}"
    return start_frame, start_time, end_time

# Função para processar cada pasta de vídeos
def process_videos(path: str, with_ads: bool = None):
    for filename in os.listdir(path):
        if not filename.lower().endswith('.mp4'):
            continue

        full_path = os.path.join(path, filename)
        raw_size = os.path.getsize(full_path) / (1024 * 1024)
        size_mb = round(raw_size, 1)  # Apenas uma casa decimal

        if with_ads is None:
            match = timecode_pattern.match(filename)
            if match:
                time_part = match.group('time')
                _, start_time, end_time = calculate_times(time_part)
                desc = 'With PiP'
            else:
                start_time, end_time, desc = None, None, 'Without PiP'

            data.append([
                full_path, start_time, end_time, desc,
                DURATION, FORMAT, CODEC, FRAME_SIZE, size_mb
            ])

        elif with_ads:
            match = timecode_pattern.match(filename)
            if match:
                time_part = match.group('time')
                _, start_time, end_time = calculate_times(time_part)
                desc = 'With PiP'
                data.append([
                    full_path, start_time, end_time, desc,
                    DURATION, FORMAT, CODEC, FRAME_SIZE, size_mb
                ])
            else:
                print(f"[ERRO] Vídeo com propaganda mal formatado: {filename}")

        else:
            match = timecode_pattern.match(filename)
            if not match:
                data.append([
                    full_path, None, None, 'Without PiP',
                    DURATION, FORMAT, CODEC, FRAME_SIZE, size_mb
                ])
            else:
                print(f"[ERRO] Vídeo sem propaganda mal formatado: {filename}")

# Processamento das pastas
process_videos(video_paths['TRAIN_COM'], True)
process_videos(video_paths['TRAIN_Sem'], False)
process_videos(video_paths['VALID'])
process_videos(video_paths['TEST'])

# Criação do DataFrame
df = pd.DataFrame(data[1:], columns=data[0])

# Salvando arquivos de forma segura
csv_path = os.path.join(OUT_DIR, "DataSet_S_K_Frame_ORG.csv")
xlsx_path = os.path.join(OUT_DIR, "DataSet_S_K_Frame_ORG.xlsx")

df.to_csv(csv_path, index=False)
df.to_excel(xlsx_path, index=False, engine='openpyxl')

print(f"Arquivos gerados com sucesso:\n  - {csv_path}\n  - {xlsx_path}")


[notice] A new release of pip is available: 25.1.1 -> 25.2
[notice] To update, run: python.exe -m pip install --upgrade pip


Arquivos gerados com sucesso:
  - C:\DataSet_SoccerKeyFrame - Org\DataSet_S_K_Frame_ORG.csv
  - C:\DataSet_SoccerKeyFrame - Org\DataSet_S_K_Frame_ORG.xlsx
